# LangChain RAG Tool — Rewritten for LangChain v1 (`langchain==1.3.11`)

This notebook keeps the **exact same task** as the original `9_LangchainRAGTool.ipynb`: build a FAISS-backed retriever over `sample.txt`, wrap it as a tool the agent can call, give the agent conversational memory, and ask it the same 3 questions — *"What is LangChain?"*, *"Who created it?"*, *"Explain LangChain's use in AI workflows."*

Only **how** it's built has changed — the legacy `initialize_agent` + `AgentType.CONVERSATIONAL_REACT_DESCRIPTION` + `RetrievalQA` + `ConversationBufferMemory` stack is replaced with [`create_agent`](https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent):

| Legacy piece | v1 replacement |
|---|---|
| `RetrievalQA.from_chain_type(llm, retriever)` wrapped in a `Tool` | A plain `@tool` function that calls `retriever.invoke(query)` and has the LLM answer from that context |
| `ConversationBufferMemory(memory_key="chat_history", ...)` | `checkpointer=InMemorySaver()` + `thread_id` |
| `initialize_agent(..., agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION, handle_parsing_errors=True)` | `create_agent(model, tools, checkpointer=...)` |
| `handle_parsing_errors=True` (needed because the old text-based ReAct agent could fail to parse the LLM's free-form output — see the "Could not parse LLM output" error in the original notebook's own Q3 run) | Not needed — native tool calling has no free-text format to fail on |
| `langchain.chat_models.ChatOpenAI` | `langchain_openai.ChatOpenAI` |
| `langchain.vectorstores.FAISS` / `langchain.embeddings.OpenAIEmbeddings` | `langchain_community.vectorstores.FAISS` / `langchain_openai.OpenAIEmbeddings` |
| `langchain.text_splitter.CharacterTextSplitter` | `langchain_text_splitters.CharacterTextSplitter` |

Reference: <https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent>


In [2]:
from IPython.display import HTML, display

html = """
<div style="
    font-family: Segoe UI;
    background: linear-gradient(135deg,#0F172A,#1E293B);
    color:white;
    padding:25px;
    border-radius:15px;
    line-height:1.8;
">

<h2 style="color:#38BDF8;">🔗 LangChain RAG Agent Summary</h2>

<p>
This project demonstrates how <b>LangChain</b> can build a simple
<b>RAG (Retrieval-Augmented Generation)</b> application.
</p>

<h3 style="color:#FBBF24;">📌 What Happens?</h3>

<ul>
<li>📄 Reads knowledge from <b>sample.txt</b></li>
<li>✂️ Splits the document into smaller chunks</li>
<li>🧠 Converts chunks into embeddings using OpenAI Embeddings</li>
<li>📚 Stores embeddings in a FAISS Vector Database</li>
<li>🔍 Retrieves relevant information when a question is asked</li>
<li>🤖 Sends retrieved content to GPT-4.1 Mini</li>
<li>💬 Returns an accurate answer based on the retrieved context</li>
<li>📝 Stores conversation history using LangGraph Memory</li>
</ul>

<h3 style="color:#34D399;">🎯 Purpose of LangChain RAG</h3>

<p>
RAG allows an AI model to answer questions using your own documents
instead of relying only on its training data.
</p>

<p>
Without RAG:
<br>❌ AI answers from general knowledge
</p>

<p>
With RAG:
<br>✅ AI searches your documents
<br>✅ Retrieves relevant content
<br>✅ Uses that content to generate accurate answers
</p>

<h3 style="color:#A78BFA;">🚀 Real-World Use Cases</h3>

<ul>
<li>Company Knowledge Base</li>
<li>PDF Question Answering</li>
<li>Customer Support Bots</li>
<li>Policy & Compliance Search</li>
<li>Document Assistants</li>
<li>Enterprise Chatbots</li>
</ul>

<p style="background:#334155;padding:10px;border-radius:8px;">
<b>In Simple Terms:</b><br>
LangChain RAG lets an AI "open a book, find the right page,
read it, and then answer your question."
</p>

</div>
"""

display(HTML(html))

In [1]:
# 📦 Install required packages (run once)
# langchain          -> Core LangChain framework
# langchain-openai   -> OpenAI integrations
# langchain-community-> Community integrations like FAISS
# langchain-text-splitters -> Text chunking utilities
# faiss-cpu          -> Vector database for similarity search
# tiktoken           -> Token counting support
# python-dotenv      -> Load API keys from .env file
# langgraph          -> Agent memory and state management
# !pip install "langchain==1.3.11" langchain-openai langchain-community langchain-text-splitters faiss-cpu tiktoken python-dotenv langgraph


# ===============================
# 1️⃣ IMPORT REQUIRED LIBRARIES
# ===============================

# Creates modern LangChain agents
from langchain.agents import create_agent

# Converts a Python function into a tool that agents can use
from langchain.tools import tool

# OpenAI LLM and embedding models
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# FAISS vector database for storing text embeddings
from langchain_community.vectorstores import FAISS

# Splits large text into smaller chunks
from langchain_text_splitters import CharacterTextSplitter

# Memory storage for conversations
from langgraph.checkpoint.memory import InMemorySaver

# Standard Python modules
import os
from dotenv import load_dotenv


# ===============================
# 2️⃣ LOAD OPENAI API KEY
# ===============================

# Load variables from .env file
load_dotenv(".env")

# Store API key in environment variable
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


# ===============================
# 3️⃣ INITIALIZE THE LLM
# ===============================

# Create ChatGPT model connection
#
# gpt-4.1-mini:
# - Fast
# - Low cost
# - Good reasoning
#
# temperature=0 means:
# - Deterministic responses
# - Less creativity
# - More consistent answers
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


# ===============================
# 4️⃣ BUILD VECTOR DATABASE
# ===============================

# Read local knowledge file
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# Split the document into smaller chunks
#
# chunk_size=300
# Maximum characters per chunk
#
# chunk_overlap=50
# Shares 50 characters with next chunk
# so important context isn't lost
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=300,
    chunk_overlap=50
)

# Create text chunks
texts = splitter.split_text(text_data)

# Convert text into vector embeddings
embedding = OpenAIEmbeddings()

# Create FAISS vector store
#
# Process:
# Text -> Embeddings -> Stored in FAISS
vectorstore = FAISS.from_texts(
    texts,
    embedding
)

# Convert vector database into retriever
#
# Retriever's job:
# User Question
#      ↓
# Find Similar Chunks
#      ↓
# Return Relevant Context
retriever = vectorstore.as_retriever()


# ===============================
# 5️⃣ CREATE RETRIEVAL TOOL
# ===============================

# @tool decorator registers this
# function as an Agent Tool
@tool
def LangChainRetriever(query: str) -> str:
    """
    Tool used to answer LangChain-related questions.

    The agent can call this tool whenever
    it needs information stored in sample.txt.
    """

    # Search vector database for relevant chunks
    docs = retriever.invoke(query)

    # Combine retrieved chunks into one context block
    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    # Ask LLM to answer strictly using retrieved context
    answer = llm.invoke(
        f"""
        Answer the question using only the context below.

        Context:
        {context}

        Question:
        {query}
        """
    )

    # Return final answer text
    return answer.content


# ===============================
# 6️⃣ CREATE MEMORY
# ===============================

# In-memory conversation storage
#
# Stores:
# User messages
# Assistant replies
# Tool calls
# Agent state
#
# Memory exists only while the program runs
checkpointer = InMemorySaver()

# Unique conversation ID
#
# All messages under this thread
# are remembered together
thread_config = {
    "configurable": {
        "thread_id": "rag-demo-1"
    }
}


# ===============================
# 7️⃣ CREATE AGENT
# ===============================

# create_agent() replaces older
# initialize_agent() approach
#
# Components:
# LLM
# + Tool
# + Memory
#
# = Intelligent Agent
agent = create_agent(
    model=llm,
    tools=[LangChainRetriever],
    checkpointer=checkpointer
)


# ===============================
# 8️⃣ ASK QUESTIONS
# ===============================

# First user question
print("1️⃣ First Question")

# Agent receives message
#
# Flow:
# User Question
#      ↓
# Agent decides whether tool is needed
#      ↓
# Tool retrieves knowledge
#      ↓
# LLM generates answer
res1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is LangChain?"
            }
        ]
    },
    thread_config
)

# Display final answer
print("Answer:", res1["messages"][-1].content)


# ===============================
# FOLLOW-UP QUESTION
# ===============================

print("\n2️⃣ Follow-up")

# Memory allows the agent to remember
# earlier conversation in same thread
res2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Who created it?"
            }
        ]
    },
    thread_config
)

print("Answer:", res2["messages"][-1].content)


# ===============================
# COMBINED REASONING QUESTION
# ===============================

print("\n3️⃣ Combined Reasoning")

# Agent uses:
# 1. Retrieved knowledge
# 2. Conversation memory
# 3. LLM reasoning
#
# to generate a richer answer
res3 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Explain LangChain's use in AI workflows."
            }
        ]
    },
    thread_config
)

print("Answer:", res3["messages"][-1].content)

C:\Users\xsanthkum\AppData\Local\Temp\ipykernel_31104\1871710873.py:27: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


1️⃣ First Question
Answer: LangChain is a framework for building applications with large language models (LLMs). If you want, I can provide more detailed information about its features and use cases.

2️⃣ Follow-up
Answer: LangChain was created by Harrison Chase.

3️⃣ Combined Reasoning
Answer: LangChain is used in AI workflows by providing a framework that supports components such as Retrieval-Augmented Generation (RAG), agents, memory, and tools. This enables developers to build applications that integrate large language models effectively, allowing for complex interactions and processes within AI systems.
